# Semana 12 — Evolução para VPC com Camadas Pública e Privada
## Prática reproduzível no AWS Academy Learner Lab

Na Semana 10, rastreamos uma requisição até a EC2 pública e distinguimos endereço, rota, Security Group, sistema operacional e contêiner. Nesta aula, vamos **redesenhar e implementar** uma arquitetura em duas camadas: uma instância frontal acessível pela internet e um backend sem endereço público.

> **Pergunta central:** como disponibilizar uma aplicação sem expor diretamente o backend?

Este caderno prioriza decisões no Console AWS, evidências e interpretação dos resultados. Os poucos comandos apresentados servem apenas para configurar os servidores Linux; não há simulação em Python.

### Objetivos de aprendizagem

Ao concluir a prática, você deverá ser capaz de:

- criar uma VPC com endereço planejado;
- separar sub-redes pública e privada;
- associar cada sub-rede à tabela de rotas correta;
- configurar Internet Gateway e NAT Gateway;
- lançar um frontend público e um backend privado;
- encadear Security Groups com privilégio mínimo;
- comprovar os fluxos de entrada, comunicação interna e saída;
- diagnosticar falhas observando rotas, regras, IPs e estado dos serviços;
- desligar e remover recursos que geram custos.

![VPC com camadas pública e privada](images/semana_12_vpc_publica_privada.png)

## 1. Arquitetura que será construída

A versão de laboratório usa uma Zona de Disponibilidade para reduzir quantidade de recursos. Ao final, discutiremos a evolução para duas AZs.

```mermaid
flowchart LR
  U[Usuário na internet] --> IGW[Internet Gateway]
  IGW --> F[Frontend Nginx<br/>sub-rede pública]
  F -->|HTTP pela rede privada| B[Backend Nginx<br/>sub-rede privada]
  B -->|saída iniciada internamente| NAT[NAT Gateway<br/>sub-rede pública]
  NAT --> IGW
```

### Plano de endereçamento e nomes

| Recurso | Nome sugerido | Configuração |
|---|---|---|
| VPC | `smartn1-vpc` | `10.20.0.0/16` |
| sub-rede pública | `smartn1-public-a` | `10.20.1.0/24` |
| sub-rede privada | `smartn1-private-a` | `10.20.11.0/24` |
| Internet Gateway | `smartn1-igw` | anexado à VPC |
| tabela pública | `smartn1-rt-public` | saída padrão para o IGW |
| NAT Gateway | `smartn1-nat-a` | na sub-rede pública |
| tabela privada | `smartn1-rt-private` | saída padrão para o NAT |
| Security Group frontal | `smartn1-sg-front` | HTTP público |
| Security Group backend | `smartn1-sg-back` | HTTP apenas do frontend |

> Use a mesma região durante toda a aula. Recursos criados em regiões diferentes não aparecerão juntos nas listas do console.

## 2. Antes de começar

### Separação entre Nuvem e DevOps

Nesta aula, as imagens Docker já estarão publicadas no Docker Hub do professor:

```text
PROFESSOR_DOCKERHUB/smartn1-backend:1.0
PROFESSOR_DOCKERHUB/smartn1-frontend:1.0
```

O professor informará o valor real de `PROFESSOR_DOCKERHUB`. O aluno não construirá Dockerfiles, não criará tags e não executará `docker push` nesta aula. Essas competências serão desenvolvidas posteriormente em DevOps.

O foco de Nuvem será:

- preparar a rede AWS;
- instalar o Docker nas EC2;
- baixar imagens prontas com `docker pull`;
- configurar a comunicação entre contêineres em máquinas distintas;
- validar isolamento, rotas e Security Groups.

### Preparação do laboratório

1. Abra o **AWS Academy Learner Lab**.
2. Clique em **Start Lab** e aguarde o indicador ficar verde.
3. Abra o **AWS Console** pelo link do laboratório.
4. Confirme a região indicada para a turma.
5. Abra o serviço **VPC**.
6. Anote ou capture a tela de recursos existentes para não remover itens do laboratório por engano.

### Cuidados importantes

- NAT Gateway e endereços IPv4 públicos podem consumir créditos do laboratório.
- Crie apenas um NAT para esta prática.
- Não reutilize a VPC padrão sem compreender os recursos existentes.
- Use os nomes sugeridos para facilitar diagnóstico e exclusão.
- Ao terminar, siga a limpeza na ordem apresentada no final do caderno.

Se o laboratório não autorizar NAT Gateway, execute até as tabelas de rotas, documente o desenho e valide a comunicação interna. Não substitua silenciosamente o recurso por uma configuração diferente.

## 3. Etapa 1 — Criar a VPC

No Console AWS:

1. Acesse **VPC → Your VPCs**.
2. Clique em **Create VPC**.
3. Em **Resources to create**, escolha **VPC only**.
4. Em **Name tag**, informe `smartn1-vpc`.
5. Em **IPv4 CIDR manual input**, informe `10.20.0.0/16`.
6. Mantenha **IPv6 CIDR block** como nenhum para esta prática.
7. Mantenha **Tenancy** como `Default`.
8. Clique em **Create VPC**.
9. Abra a VPC criada e confirme que **DNS resolution** e **DNS hostnames** estão habilitados. Caso não estejam, use **Actions → Edit VPC settings**.

### O que deve aparecer

- estado `Available`;
- CIDR IPv4 `10.20.0.0/16`;
- um identificador começando por `vpc-`;
- uma tabela de rotas principal e uma ACL padrão criadas automaticamente.

### Como interpretar

A VPC define o espaço de rede. Ela ainda não possui caminho para a internet. Criar a VPC não torna nenhum servidor público.

## 4. Etapa 2 — Criar as sub-redes

1. Acesse **VPC → Subnets → Create subnet**.
2. Escolha `smartn1-vpc`.
3. Escolha uma Zona de Disponibilidade e anote-a, por exemplo `us-east-1a`.
4. Crie a primeira sub-rede:
   - nome: `smartn1-public-a`;
   - IPv4 subnet CIDR: `10.20.1.0/24`.
5. Na mesma tela, use **Add new subnet** e crie:
   - nome: `smartn1-private-a`;
   - mesma Zona de Disponibilidade;
   - IPv4 subnet CIDR: `10.20.11.0/24`.
6. Clique em **Create subnet**.
7. Selecione `smartn1-public-a` e use **Actions → Edit subnet settings**.
8. Habilite **Enable auto-assign public IPv4 address** apenas na sub-rede pública.
9. Confirme que essa opção permanece desabilitada na sub-rede privada.

### O que deve aparecer

As duas sub-redes pertencem à mesma VPC, não se sobrepõem e possuem identificadores `subnet-...` distintos.

### Como interpretar

O nome “public” ainda é apenas um rótulo. A sub-rede se tornará efetivamente pública quando for associada a uma tabela com rota para o Internet Gateway. A opção de IP público, sozinha, também não cria a rota.

## 5. Etapa 3 — Criar e anexar o Internet Gateway

1. Acesse **VPC → Internet gateways**.
2. Clique em **Create internet gateway**.
3. Nomeie como `smartn1-igw` e confirme.
4. Selecione o gateway criado.
5. Use **Actions → Attach to a VPC**.
6. Escolha `smartn1-vpc` e confirme.

### O que deve aparecer

O estado do gateway deve indicar que ele está anexado à `smartn1-vpc`.

### Como interpretar

O Internet Gateway é a borda entre a VPC e a internet, mas ainda falta uma tabela de rotas apontando para ele. O gateway anexado, sem rota, não cria conectividade externa.

## 6. Etapa 4 — Criar a tabela de rotas pública

1. Acesse **VPC → Route tables → Create route table**.
2. Nome: `smartn1-rt-public`.
3. VPC: `smartn1-vpc`.
4. Crie a tabela.
5. Abra a aba **Routes → Edit routes**.
6. Preserve a rota `10.20.0.0/16 → local`.
7. Adicione `0.0.0.0/0` com alvo **Internet Gateway** e escolha `smartn1-igw`.
8. Salve as alterações.
9. Abra **Subnet associations → Edit subnet associations**.
10. Marque apenas `smartn1-public-a` e salve.

### Resultado esperado

| Destino | Alvo |
|---|---|
| `10.20.0.0/16` | `local` |
| `0.0.0.0/0` | `smartn1-igw` |

### Como interpretar

- tráfego destinado à própria VPC usa a rota local;
- qualquer outro destino IPv4 usa o Internet Gateway;
- a associação explícita transforma `smartn1-public-a` em sub-rede pública;
- `smartn1-private-a` não deve aparecer nas associações desta tabela.

## 7. Etapa 5 — Criar o NAT Gateway

1. Acesse **VPC → NAT gateways → Create NAT gateway**.
2. Nome: `smartn1-nat-a`.
3. Subnet: `smartn1-public-a`.
4. Connectivity type: **Public**.
5. Em **Elastic IP allocation ID**, clique em **Allocate Elastic IP**.
6. Clique em **Create NAT gateway**.
7. Aguarde o estado mudar de `Pending` para `Available` antes de configurar a rota privada.

### O que deve aparecer

- NAT dentro da sub-rede pública;
- Elastic IP associado;
- estado `Available`;
- identificador começando por `nat-`.

### Como interpretar

O NAT permite que recursos privados iniciem conexões IPv4 com a internet e recebam as respectivas respostas. Ele não oferece um endereço pelo qual usuários externos iniciem conexão com o backend.

> Um NAT público precisa estar em uma sub-rede que tenha rota ao Internet Gateway. Colocá-lo na sub-rede privada quebra o fluxo esperado.

## 8. Etapa 6 — Criar a tabela de rotas privada

1. Volte a **VPC → Route tables → Create route table**.
2. Nome: `smartn1-rt-private`.
3. VPC: `smartn1-vpc`.
4. Crie a tabela.
5. Em **Routes → Edit routes**, preserve a rota local.
6. Adicione `0.0.0.0/0` com alvo **NAT Gateway** e escolha `smartn1-nat-a`.
7. Salve.
8. Em **Subnet associations**, associe apenas `smartn1-private-a`.

### Resultado esperado

| Destino | Alvo |
|---|---|
| `10.20.0.0/16` | `local` |
| `0.0.0.0/0` | `smartn1-nat-a` |

### Como interpretar

A tabela privada não aponta diretamente ao Internet Gateway. O backend pode iniciar uma conexão externa passando pelo NAT, mas não recebe conexão iniciada por um usuário da internet.

### Verificação cruzada obrigatória

Na lista de sub-redes, confirme:

- `smartn1-public-a` → `smartn1-rt-public`;
- `smartn1-private-a` → `smartn1-rt-private`.

Uma associação errada é uma das causas mais comuns de falha nesta prática.

## 9. Etapa 7 — Criar Security Groups encadeados

### Security Group do frontend

1. Acesse **EC2 → Security Groups → Create security group**.
2. Nome: `smartn1-sg-front`.
3. VPC: `smartn1-vpc`.
4. Entrada HTTP TCP/80 com origem `0.0.0.0/0`.
5. Para administração, prefira **EC2 Instance Connect** ou **Session Manager**. Se o laboratório exigir SSH, libere TCP/22 somente para **My IP**, nunca para toda a internet.
6. Mantenha saída liberada para a prática.

### Security Group do backend

1. Crie `smartn1-sg-back` na mesma VPC.
2. Adicione entrada **Custom TCP / 8000**.
3. Na origem, selecione **Custom** e informe o Security Group `smartn1-sg-front`, não um CIDR público.
4. Não crie regra SSH pública para o backend.
5. Mantenha saída liberada para o `docker pull` realizado através do NAT.

### Como interpretar

O frontend aceita HTTP da internet na porta 80. O backend aceita a API na porta 8000 somente de interfaces associadas ao Security Group frontal. Mesmo dentro da VPC, outra instância sem `smartn1-sg-front` não recebe automaticamente essa autorização.

## 10. Etapa 8 — Lançar o backend privado e baixar a imagem

### 10.1 Criar a instância

1. Acesse **EC2 → Instances → Launch instances**.
2. Nome: `smartn1-backend`.
3. AMI: **Amazon Linux 2023**, se disponível.
4. Tipo: uma opção pequena permitida pelo Learner Lab, como `t3.micro` ou equivalente disponível.
5. Key pair: escolha `vockey` somente se exigido pelo ambiente.
6. Em **Network settings → Edit**:
   - VPC: `smartn1-vpc`;
   - Subnet: `smartn1-private-a`;
   - Auto-assign public IP: **Disable**;
   - Security Group: `smartn1-sg-back`.
7. Se o laboratório disponibilizar um instance profile compatível, associe `LabInstanceProfile`/`LabRole` para permitir gerenciamento autorizado.
8. Lance a instância e aguarde os checks ficarem `2/2 passed`.
9. Anote o **Private IPv4 address**.

### 10.2 Instalar Docker e executar a imagem do professor

Conecte por Session Manager, quando disponível, ou pelo método autorizado no laboratório. Execute:

```bash
sudo dnf update -y
sudo dnf install -y docker
sudo systemctl enable --now docker
sudo usermod -aG docker ec2-user
```

Atualize a sessão do grupo ou use `sudo docker` nesta primeira execução. Substitua `PROFESSOR_DOCKERHUB` pelo usuário informado:

```bash
sudo docker pull PROFESSOR_DOCKERHUB/smartn1-backend:1.0

sudo docker run -d \
  --name smartn1-backend \
  --restart unless-stopped \
  -p 8000:8000 \
  PROFESSOR_DOCKERHUB/smartn1-backend:1.0
```

Valide localmente:

```bash
sudo docker ps
sudo docker logs smartn1-backend
curl http://127.0.0.1:8000/health
curl http://127.0.0.1:8000/status
```

### O que deve aparecer

- EC2 com IPv4 privado dentro de `10.20.11.0/24` e sem IPv4 público;
- imagem `smartn1-backend:1.0` baixada do namespace do professor;
- contêiner com estado `Up` e, após alguns segundos, `healthy`;
- `/health` respondendo `healthy`;
- `/status` informando `layer: private-subnet`.

### Como interpretar

O `docker pull` bem-sucedido comprova a saída da sub-rede privada pelo NAT. O backend não precisa receber conexão da internet para baixar a imagem: ele inicia a conexão com o Docker Hub e recebe a resposta pelo mesmo fluxo.

## 11. Etapa 9 — Lançar o frontend público e conectar ao backend

### 11.1 Criar a instância

1. Inicie outra instância EC2.
2. Nome: `smartn1-frontend`.
3. Use a mesma AMI e tipo compatível.
4. Em rede:
   - VPC: `smartn1-vpc`;
   - Subnet: `smartn1-public-a`;
   - Auto-assign public IP: **Enable**;
   - Security Group: `smartn1-sg-front`.
5. Lance a instância e aguarde `2/2 checks passed`.
6. Anote o **Public IPv4 address**.

### 11.2 Instalar Docker e executar a imagem do professor

Conecte à instância pelo método autorizado e instale o Docker:

```bash
sudo dnf update -y
sudo dnf install -y docker
sudo systemctl enable --now docker
sudo usermod -aG docker ec2-user
```

Substitua `PROFESSOR_DOCKERHUB` e `IP_PRIVADO_BACKEND`:

```bash
sudo docker pull PROFESSOR_DOCKERHUB/smartn1-frontend:1.0

sudo docker run -d \
  --name smartn1-frontend \
  --restart unless-stopped \
  -p 80:8080 \
  -e BACKEND_HOST=IP_PRIVADO_BACKEND \
  -e BACKEND_PORT=8000 \
  PROFESSOR_DOCKERHUB/smartn1-frontend:1.0
```

Valide o contêiner:

```bash
sudo docker ps
sudo docker logs smartn1-frontend
curl http://127.0.0.1/health
```

### Como interpretar

A variável `BACKEND_HOST` informa ao proxy Nginx o endereço privado do backend. O navegador chama `/api/status` no frontend público; o frontend encaminha internamente para `IP_PRIVADO_BACKEND:8000/status`. O navegador nunca precisa alcançar a subnet privada.

## 12. Etapa 10 — Validar os três fluxos

### Teste A — Entrada pública e comunicação entre camadas

Abra no navegador:

```text
http://IP_PUBLICO_FRONTEND
```

**Resultado esperado:** a página Smart N1 apresenta “Backend privado respondeu” e exibe serviço, versão, hostname, camada e horário retornados pela API.

**Interpretação:** o navegador alcançou o contêiner frontend; o proxy alcançou o contêiner backend pela rede privada; a resposta percorreu o caminho inverso.

### Teste B — Isolamento do backend

Na instância `smartn1-backend`, confirme que **Public IPv4 address** está vazio. Não existe URL pública direta para a API.

**Interpretação:** a API é consumida pelo frontend dentro da VPC e não foi exposta diretamente.

### Teste C — Security Group encadeado

Remova temporariamente a entrada TCP/8000 de `smartn1-sg-back` ou substitua sua origem. Atualize a página.

**Resultado esperado:** o frontend continua abrindo, mas indica falha ao alcançar o backend. Restaure imediatamente a regra TCP/8000 com origem `smartn1-sg-front`.

**Interpretação:** a rota local continuou existindo; a autorização entre as camadas foi removida.

### Teste D — Saída privada pelo NAT

No backend, execute:

```bash
sudo docker pull PROFESSOR_DOCKERHUB/smartn1-backend:1.0
```

**Resultado esperado:** a verificação da imagem é concluída ou informa que ela já está atualizada.

Depois, remova temporariamente a rota privada `0.0.0.0/0 → NAT`, tente o pull novamente e restaure a rota.

**Interpretação:** o backend mantém comunicação local com a VPC, mas perde o caminho para iniciar conexões com o Docker Hub.

## 13. Diagnóstico orientado por sintomas

| Sintoma | Verificação inicial | Interpretação provável |
|---|---|---|
| frontend não abre | IP público, rota pública e SG frontal | falha antes do frontend |
| frontend retorna 502/504 | Nginx backend, SG backend e IP configurado | frontend chegou, backend não respondeu |
| backend não instala Nginx | NAT disponível, rota privada e saída do SG | falha de saída da subnet privada |
| backend possui IP público | configuração da interface/subnet | isolamento incorreto |
| NAT permanece `Pending`/`Failed` | subnet, EIP e limites do lab | rota privada ainda não funcionará |
| acesso direto ao backend funciona pela internet | IP público ou regra indevida | requisito de isolamento violado |

### Ordem recomendada de investigação

1. confirme região e VPC;
2. confirme subnet e associação da tabela;
3. confirme rota e estado do alvo;
4. confirme IP público/privado;
5. confirme Security Group;
6. confirme serviço e porta no sistema operacional;
7. confirme configuração do proxy.

## 14. Como seria a evolução para produção

O laboratório demonstra o conceito, mas ainda possui pontos únicos de falha. Uma arquitetura mais resiliente usaria:

- duas sub-redes públicas em AZs diferentes;
- Application Load Balancer público em vez de uma EC2 frontal única;
- duas sub-redes privadas de aplicação;
- Auto Scaling ou ECS distribuído entre AZs;
- NAT por AZ, conforme requisito de disponibilidade e orçamento;
- banco gerenciado em sub-redes isoladas e configuração Multi-AZ;
- VPC Endpoints para reduzir tráfego pelo NAT;
- HTTPS com certificado gerenciado;
- logs, métricas, alarmes e backups.

| Provedor | Rede | Saída privada | Firewall do recurso | Borda HTTP |
|---|---|---|---|---|
| AWS | VPC | NAT Gateway | Security Group | Application Load Balancer |
| Azure | Virtual Network | NAT Gateway | NSG | Application Gateway |
| Google Cloud | VPC Network | Cloud NAT | VPC Firewall Rules | Application Load Balancer |
| Oracle Cloud | VCN | NAT Gateway | NSG/Security List | Load Balancer |

Os nomes são equivalentes conceituais; comportamento, escopo e cobrança variam entre provedores.

## 15. Exercícios práticos

### Exercício 1 — Evidências da rede

Capture e identifique:

- VPC e CIDR;
- duas sub-redes;
- associação de cada tabela de rotas;
- rotas pública e privada;
- NAT no estado `Available`.

Explique o que cada captura comprova.

### Exercício 2 — Caminho da requisição

Desenhe e numere o percurso do navegador até o backend. Indique onde ocorrem roteamento, autorização e proxy HTTP.

### Exercício 3 — Falha controlada

Remova temporariamente a autorização do Security Group do backend, observe a resposta do frontend e restaure a configuração. Explique por que a rota local não foi suficiente.

### Exercício 4 — Diagnóstico do NAT

Retire temporariamente a rota privada para o NAT. Explique quais fluxos deixam de funcionar e quais continuam disponíveis. Restaure a rota.

### Exercício 5 — Privilégio mínimo

Revise os dois Security Groups e elimine regras que não sejam necessárias. Justifique cada porta e origem restante.

### Exercício 6 — Evolução para duas AZs

Amplie o diagrama adicionando sub-redes `10.20.2.0/24` e `10.20.12.0/24` em outra AZ. Posicione ALB, segunda aplicação e NAT de acordo com o requisito de disponibilidade.

**Entregável:** relatório curto com arquitetura, evidências, testes executados, uma falha diagnosticada e proposta de evolução.

## 16. Limpeza obrigatória do laboratório

Remova recursos nesta ordem para evitar dependências e consumo de créditos:

1. encerre `smartn1-frontend` e `smartn1-backend`;
2. exclua `smartn1-nat-a` e aguarde a exclusão;
3. libere o Elastic IP que não estiver mais associado;
4. exclua os Security Groups personalizados;
5. remova associações e exclua tabelas de rotas personalizadas;
6. exclua as duas sub-redes;
7. desanexe e exclua `smartn1-igw`;
8. exclua `smartn1-vpc`;
9. confirme que não restaram NAT Gateways ou Elastic IPs;
10. encerre a sessão com **End Lab**.

Não exclua recursos padrão ou de outros projetos. Confirme sempre a tag `smartn1-...` antes de remover.

## Bibliografia e documentação oficial

- AWS. **Create a VPC with private subnets and NAT gateways**. <https://docs.aws.amazon.com/vpc/latest/userguide/create-a-vpc-with-private-subnets-and-nat-gateways.html>.
- AWS. **Example: VPC with servers in private subnets and NAT**. <https://docs.aws.amazon.com/vpc/latest/userguide/vpc-example-private-subnets-nat.html>.
- AWS. **Subnet route tables**. <https://docs.aws.amazon.com/vpc/latest/userguide/subnet-route-tables.html>.
- AWS. **NAT gateways**. <https://docs.aws.amazon.com/vpc/latest/userguide/vpc-nat-gateway.html>.
- AWS. **Security groups for your VPC**. <https://docs.aws.amazon.com/vpc/latest/userguide/vpc-security-groups.html>.
- AWS. **User data and shell scripts**. <https://docs.aws.amazon.com/AWSEC2/latest/UserGuide/user-data.html>.
- KUROSE, J.; ROSS, K. *Redes de Computadores e a Internet*. 8. ed. Bookman, 2021.
- VERAS, M. *Arquitetura de nuvem: Amazon Web Services*. Brasport, 2013.